# Tier 3.1: GPU setup and first run

The GPU tier keeps the bit-vector layout of tier 2 and moves it into GPU memory, where CUDA kernels count
candidates. This notebook lists what the GPU tier needs, detects the devices, runs the smallest possible GPU mine,
and checks the GPU result against tiers 1 and 2.

**What you will learn**

- the driver, CUDA and package requirements;
- how the library detects GPUs, and how to check the kernels compile;
- the smallest GPU call and its equality check against the CPU tiers;
- what happens when you ask for the GPU on a machine without one.

**Prerequisites**: tiers 1 and 2 ([tier 2, notebook 3](../tier2-rust-pyo3/03-same-results-and-speed.ipynb)).

**How to read the outputs.** The setup cell prints the hardware the committed outputs came from.
Cells marked *GPU* print `skipped: no CUDA device` on a machine without one; every other cell runs on the CPU.
The checklist at the end lists what to confirm after running the notebook on a GPU.

## 0. Setup and device check

The setup cell sets `SKIP_GPU` when CuPy is missing or no CUDA device is usable, and `SKIP_MULTI` for fewer than two devices.

In [1]:
import os
import warnings

os.environ.setdefault("LOGURU_LEVEL", "WARNING")
warnings.filterwarnings("ignore", message="IProgress not found")

import datetime
import platform
import re
from pathlib import Path

import polars as pl

import et_miner
from et_miner import apriori

DATA = Path("../data")
REPO = Path("../..").resolve()
N_GPUS = et_miner.get_gpu_count()
SKIP_GPU = not et_miner.has_cupy()  # CuPy installed and at least one CUDA device usable
SKIP_MULTI = N_GPUS < 2


def hardware() -> str:
    cpu = platform.processor() or platform.machine()
    label = f"{cpu}, {os.cpu_count()} logical CPUs, Python {platform.python_version()}, {datetime.date.today()}"
    if not SKIP_GPU:
        import cupy as cp

        names = []
        for d in range(N_GPUS):
            name = cp.cuda.runtime.getDeviceProperties(d)["name"]
            names.append(name.decode() if isinstance(name, bytes) else name)
        label += f" | {N_GPUS} GPU(s): {', '.join(names)} | CUDA runtime {cp.cuda.runtime.runtimeGetVersion()}"
    return label


def show(rel_path: str, pattern: str, n_lines: int) -> None:
    """Print n_lines of a repository file, starting at the first line matching pattern."""
    lines = (REPO / rel_path).read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if re.search(pattern, line))
    print(f"{rel_path}:{start + 1}")
    for i in range(start, min(start + n_lines, len(lines))):
        print(f"{i + 1:5d}  {lines[i]}")


print(hardware())
if SKIP_GPU:
    print("No usable CUDA device: GPU cells print a skip line instead of running.")
elif SKIP_MULTI:
    print("One CUDA device: multi-GPU cells print a skip line instead of running.")

x86_64, 4 logical CPUs, Python 3.11.15, 2026-09-23
No usable CUDA device: GPU cells print a skip line instead of running.


## 1. Requirements

| Need | Detail | Source |
|---|---|---|
| NVIDIA GPU | the kernel sources use only intrinsics available from compute capability 6.0 (sm_60) on | header of `src/et_miner/gpu/kernels/_src/shared_tiled.cu` |
| Driver with CUDA 12 support | CuPy's `cupy-cuda12x` wheel targets CUDA 12 | `pyproject.toml`, `[project.optional-dependencies] gpu` |
| `cupy-cuda12x[ctk]` | CuPy plus the CUDA header wheels that NVRTC needs to compile kernels at first use | same |
| `nvidia-nccl-cu12` | NCCL, used to sum counts across GPUs | same |
| Rust extension | optional; faster candidate grouping and pruning on the host | [tier 2, notebook 1](../tier2-rust-pyo3/01-build-and-install.md) |

No CUDA toolkit or `nvcc` is needed: kernels are plain CUDA C sources compiled on the device at first use.
Install the extra into the project environment with

```bash
uv sync --inexact --extra gpu
```

`--inexact` keeps the Rust extension installed (see tier 2, notebook 1).
The next cell reads the extra from `pyproject.toml`, so the list above can be checked against HEAD.

In [2]:
import tomllib

pyproject = tomllib.loads((Path("../..") / "pyproject.toml").read_text())
for extra, packages in pyproject["project"]["optional-dependencies"].items():
    print(f"{extra}: {packages}")

gpu: ['cupy-cuda12x[ctk]>=13.0', 'nvidia-nccl-cu12>=2.18.1,<3']
gcs: ['google-cloud-storage>=2.14']


The `gpu` extra holds exactly CuPy with the CUDA headers and NCCL.

## 2. What the library detects

`src/et_miner/backends.py` is the single place that probes for CuPy and the Rust extension.
The next cell prints the public probes and runs `python -m et_miner info` (the same as `et-miner info`), the command-line view of the same checks.

In [3]:
import subprocess
import sys

print("HAS_GPU (CuPy importable):   ", et_miner.HAS_GPU)
print("has_cupy() (CuPy + a device):", et_miner.has_cupy())
print("get_gpu_count():             ", et_miner.get_gpu_count())
print("HAS_MULTI_GPU:               ", et_miner.HAS_MULTI_GPU)
print("HAS_RUST:                    ", et_miner.HAS_RUST)
info = subprocess.run([sys.executable, "-m", "et_miner", "info"], capture_output=True, text=True,
                      env={**os.environ, "LOGURU_LEVEL": "WARNING"})
print(info.stdout)

HAS_GPU (CuPy importable):    False
has_cupy() (CuPy + a device): False
get_gpu_count():              0
HAS_MULTI_GPU:                False
HAS_RUST:                     True


et-miner 0.2.0
Dependencies:
  polars: 1.43.2
  tqdm: installed
Backends:
  rust: 0.3.0
  cupy: not installed (0 GPUs)
Configuration:
  min_support: 0.01
  chunk_size: 100000



`HAS_GPU` only says CuPy imports; `has_cupy()` also needs a working device, and that is the probe to gate GPU work on.

## 3. The kernels and the on-device self-check

Every CUDA kernel is registered in `src/et_miner/gpu/kernels/loader.py:_KERNEL_FILES`, which maps a kernel name
to its source file under `gpu/kernels/_src/`. The next cell lists the registry; it needs no GPU.

In [4]:
from collections import defaultdict

from et_miner.gpu.kernels.loader import _KERNEL_FILES

by_file = defaultdict(list)
for name, source in _KERNEL_FILES.items():
    by_file[source].append(name)
for source in sorted(by_file):
    print(f"{source:28s} {', '.join(by_file[source])}")

bitvec_extract_tids.cu       bitvec_extract_tids
bootstrap_copy.cu            bootstrap_copy
compact_threshold.cu         compact_threshold
csr_to_bitvec.cu             csr_to_bitvec
csr_warp.cu                  csr_count_range, csr_count_gather, csr_write_gather
decode_candidates.cu         decode_candidates_gpu
fill_row_ids.cu              fill_row_ids
itemset_count.cu             count_itemset_fused, count_itemsets_batch
k3plus_dense.cu              count_k3plus_dense
k3plus_fullyfused.cu         count_k3plus_from_groups
k3plus_fused.cu              count_itemsets_fused_k3plus
k3plus_gpu_resident.cu       count_k3plus_gpu_resident
pairs_k2.cu                  count_pairs_fused_k2
pairs_k2_dense.cu            count_pairs_k2_dense
shared_tiled.cu              count_shared_tiled_dense, count_shared_tiled_fused


On a GPU machine, `bench/selfcheck.py` compiles every registered kernel on every device and launches the
critical ones on tiny data; exit code 0 means the device is ready. The next cell (*GPU*) runs it.

In [5]:
if SKIP_GPU:
    print("skipped: no CUDA device")
else:
    check = subprocess.run([sys.executable, "bench/selfcheck.py"], capture_output=True, text=True, cwd="../..")
    print(check.stdout[-4000:])
    print("exit code", check.returncode)

skipped: no CUDA device


A non-zero exit code here means a kernel did not compile or launch; `bench/README.md` has a troubleshooting section.

## 4. The smallest GPU run

`use_gpu=True` is the only change to the call. The next cell (*GPU*) mines the hand-made table on the GPU
and compares it with tier 1. The GPU cells use integer item ids (`a`..`f` become 0..5), because
`src/et_miner/gpu/mining.py:_build_results_from_gpu` maps columns back through an integer array.

In [6]:
toy_str = pl.read_parquet(DATA / "toy_8x6.parquet")
code = {name: i for i, name in enumerate("abcdef")}
toy = toy_str.select(pl.col("items").list.eval(pl.element().replace_strict(code, return_dtype=pl.Int64)))
tier1_toy = apriori(toy, min_support=0.25, sparse=False)
print("tier 1:", tier1_toy.height, "itemsets")
if SKIP_GPU:
    print("skipped: no CUDA device")
else:
    gpu_toy = apriori(toy, min_support=0.25, use_gpu=True)
    print(gpu_toy.sort(pl.col("itemset").list.len(), "itemset"))
    assert set(map(tuple, gpu_toy["itemset"].to_list())) == set(map(tuple, tier1_toy["itemset"].to_list()))
    print("GPU == tier 1 on the toy table")

tier 1: 13 itemsets
skipped: no CUDA device


On a GPU machine the cell prints the GPU result and asserts it has the same itemsets as tier 1.

## 5. Same answer as tiers 1 and 2 on the shared sample

The next cell mines `smoke.parquet` with tier 1 and with the whole-loop Rust route of tier 2 (both on the CPU),
then (*GPU*) with `use_gpu=True`, and asserts all three give the same itemsets and counts.

In [7]:
import numpy as np

from et_miner import apriori_from_csr

smoke = pl.read_parquet(DATA / "smoke.parquet")
N, S = smoke.height, 0.01


def counted(result):
    return {(tuple(s), round(v * N)) for s, v in zip(result["itemset"].to_list(), result["support"].to_list())}


tier1 = counted(apriori(smoke, min_support=S, sparse=False))
rows = smoke["items"].to_list()  # the sample's item ids are 0..n-1, sorted and unique within each row
indptr = np.cumsum([0] + [len(r) for r in rows]).astype(np.int64)
indices = np.concatenate([np.asarray(r, dtype=np.int64) for r in rows])
its, cnt = apriori_from_csr(indptr, indices, N, int(indices.max()) + 1, S, 0)
tier2 = {(tuple(s), int(c)) for s, c in zip(its, cnt)}
assert tier1 == tier2
print("tier 1 == tier 2:", len(tier1), "itemsets")
if SKIP_GPU:
    print("GPU: skipped: no CUDA device")
else:
    tier3 = counted(apriori(smoke, min_support=S, use_gpu=True))
    assert tier3 == tier1
    print("tier 3 (use_gpu=True) == tier 1 == tier 2:", len(tier3), "itemsets")

tier 1 == tier 2: 694 itemsets
GPU: skipped: no CUDA device


The CPU tiers agree; on a GPU machine the assertion extends the chain to the GPU tier.
The repository enforces the full chain, including multi-GPU routes and efficient-apriori, in `tests/test_tier_equivalence.py`.

## 6. Asking for the GPU without one

The GPU path does not fall back to the CPU silently. The next cell shows what `use_gpu=True` raises when CuPy is
missing; on a GPU machine it prints that the call succeeded instead.

In [8]:
try:
    apriori(toy, min_support=0.25, use_gpu=True)
    print("use_gpu=True ran on the GPU")
except Exception as err:  # the error type depends on what is missing
    print(f"{type(err).__name__}: {err}")

RuntimeError: CuPy not available


Without CuPy the call stops with an error instead of returning a CPU result, so a GPU benchmark cannot quietly run on the CPU.

## Summary and next step

- The GPU tier needs an NVIDIA driver with CUDA 12 and the `gpu` extra; kernels compile on first use.
- Gate GPU work on `et_miner.has_cupy()`; check kernels with `bench/selfcheck.py`.
- `use_gpu=True` returns the same itemsets and counts as tiers 1 and 2.

**Checklist when you run this notebook on a GPU**

- [ ] Setup cell prints the GPU name and CUDA runtime version.
- [ ] Section 3: `bench/selfcheck.py` exits with code 0.
- [ ] Section 4: the toy result prints and the assertion passes.
- [ ] Section 5: the line `tier 3 (use_gpu=True) == tier 1 == tier 2` prints.
- [ ] Section 6: prints `use_gpu=True ran on the GPU`.

Next: [02-gpu-resident-mining](02-gpu-resident-mining.ipynb).